In [10]:
"""Phase‑2 • Sub‑Path‑Based Spatiotemporal Clustering
--------------------------------------------------
Reads trajectories saved in Phase‑1, mines shared sub‑paths of length K,
and creates clusters satisfying α (size) and β (time distance).

Output: phase2_clusters.pkl.gz
"""


import gzip, pickle, collections, time
from pathlib import Path
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from tqdm import tqdm

In [12]:
# ───────────────────────────────────────────────────────────
# 2. Load trajectories (≈ seconds)
# ───────────────────────────────────────────────────────────
print("🔹  Loading Phase‑1 trajectories …")
with gzip.open(PHASE1_PKL_GZ, "rb") as f:
    payload      = pickle.load(f)
    trajectories = payload["data"]

print(f"   → {len(trajectories):,} trajectories loaded")

# Convert to DataFrame once for convenience
trip_df = pd.DataFrame(trajectories, dtype=object)
trip_df["timestamp"] = pd.to_datetime(trip_df["timestamp"], errors="coerce")
trip_df.dropna(subset=["timestamp"], inplace=True)
trip_df["trip_len"]  = trip_df["path"].str.len()
trip_df.reset_index(drop=True, inplace=True)      # compact idx 0…n‑1

N_TRIPS = len(trip_df)
print(f"   → {N_TRIPS:,} trajectories after clean‑up\n")

# Pre‑compute numpy arrays for fast lookup
timestamps = trip_df["timestamp"].astype("int64").values  # ns since epoch
trip_len   = trip_df["trip_len"].values

# rank_by_time[i] == ordinal rank of trip i by timestamp  (for O(1) sort key)
sorted_idx = np.argsort(timestamps)
rank_by_time = np.empty_like(sorted_idx)
rank_by_time[sorted_idx] = np.arange(N_TRIPS, dtype=np.int32)


🔹  Loading Phase‑1 trajectories …
   → 495,304 trajectories loaded
   → 189,221 trajectories after clean‑up



In [13]:
# ───────────────────────────────────────────────────────────
# 3. Pass 1  •  Build map: sub‑path → list(trip indices)
# ───────────────────────────────────────────────────────────
print("🔹  Mining sub‑paths … (pass 1)")
subpath_map: dict[tuple, list] = collections.defaultdict(list)

for i, path in enumerate(trip_df["path" ]):
    # iterate contiguous sub‑sequences of length K_SUBLEN
    for j in range(0, len(path) - K_SUBLEN + 1):
        sub = tuple(path[j:j + K_SUBLEN])
        subpath_map[sub].append(i)

# Prune low‑support sub‑paths early
freq_subpaths = {s: idxs for s, idxs in subpath_map.items() if len(idxs) >= ALPHA_MIN}
print(f"   → distinct sub‑paths len {K_SUBLEN}: {len(freq_subpaths):,} (support ≥ {ALPHA_MIN})\n")
subpath_map.clear()   # free memory


🔹  Mining sub‑paths … (pass 1)
   → distinct sub‑paths len 4: 27,516 (support ≥ 3)



In [17]:

# ───────────────────────────────────────────────────────────
# 4. Utility to build clusters for one sub‑path
# ───────────────────────────────────────────────────────────

def clusters_from_subpath(sub, idxs):
    clusters_local = []
    # Sort trip indices by timestamp rank (fast O(n log n) w/ cheap key)
    idxs_sorted = sorted(idxs, key=rank_by_time.__getitem__)
    group = []
    base_t = None
    for i in idxs_sorted:
        t_i = timestamps[i]
        if base_t is None:
            base_t = t_i
            group.append(i)
            continue
        if t_i - base_t <= THETA_TIME * 1_000_000_000:  # ns
            group.append(i)
        else:
            if len(group) >= ALPHA_MIN:
                gamma = len(sub) / trip_len[group].mean()
                if gamma >= GAMMA_MIN:
                    clusters_local.append(dict(
                        cluster_id = None,   # fill later
                        rep_subpath= list(sub),
                        gamma      = round(float(gamma), 3),
                        members    = group.copy()  # store indices
                    ))
            base_t = t_i
            group  = [i]
    # tail group
    if len(group) >= ALPHA_MIN:
        gamma = len(sub) / trip_len[group].mean()
        if gamma >= GAMMA_MIN:
            clusters_local.append(dict(
                cluster_id = None,
                rep_subpath= list(sub),
                gamma      = round(float(gamma), 3),
                members    = group.copy()
            ))
    return clusters_local


In [18]:

# ───────────────────────────────────────────────────────────
# 5. Pass 2  •  Build clusters  (optionally in parallel)
# ───────────────────────────────────────────────────────────
print("🔹  Building clusters … (pass 2)")
start = time.time()

iterator = freq_subpaths.items()
if N_JOBS > 1:
    results = Parallel(n_jobs=N_JOBS, batch_size=256, verbose=5)(
        delayed(clusters_from_subpath)(sub, idxs) for sub, idxs in iterator)
else:
    results = [clusters_from_subpath(sub, idxs) for sub, idxs in tqdm(iterator)]

clusters = [c for sublist in results for c in sublist]  # flatten

# assign incremental cluster IDs
for k, c in enumerate(clusters):
    c["cluster_id"] = f"C{k:06d}"
    # replace member indices by lightweight dicts for output
    c["members"] = [
        dict(trip_id=trip_df.at[i, "trip_id"], timestamp=str(trip_df.at[i, "timestamp"]))
        for i in c["members"]
    ]

print(f"   → {len(clusters):,} clusters formed  (elapsed {(time.time()-start)/60:.1f} min)\n")

# ───────────────────────────────────────────────────────────
# 6. Save clusters
# ───────────────────────────────────────────────────────────
with gzip.open(OUT_PKL_GZ, "wb") as f:
    pickle.dump(clusters, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"✅  Clusters written → {OUT_PKL_GZ}  (size {Path(OUT_PKL_GZ).stat().st_size/1e6:.1f} MB)")


🔹  Building clusters … (pass 2)


[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done 528 tasks      | elapsed:    4.1s
[Parallel(n_jobs=8)]: Done 14352 tasks      | elapsed:   11.3s
[Parallel(n_jobs=8)]: Done 27060 tasks      | elapsed:   12.2s
[Parallel(n_jobs=8)]: Done 27516 out of 27516 | elapsed:   12.3s finished


   → 1,749 clusters formed  (elapsed 0.2 min)

✅  Clusters written → phase2_clusters.pkl.gz  (size 0.1 MB)
